<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model08_Credit_Card_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Drive and import required libraries
from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define data and results paths
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [ ]:
# Load application_train.csv
application = pd.read_csv(DATA_PATH + "application_train.csv")
print("Application shape:", application.shape)
# Load previous_application.csv
prev = pd.read_csv(DATA_PATH + "previous_application.csv")
print("Previous application shape:", prev.shape)
# Load bureau.csv
bureau = pd.read_csv(DATA_PATH + "bureau.csv")
print("Bureau shape:", bureau.shape)
# Load bureau_balance.csv
bureau_balance = pd.read_csv(DATA_PATH + "bureau_balance.csv")
print("Bureau balance shape:", bureau_balance.shape)
# Load POS_CASH_balance.csv with compact dtypes
pos_dtype = {
    "SK_ID_PREV": np.uint32, "SK_ID_CURR": np.uint32, "MONTHS_BALANCE": np.int32,
    "SK_DPD": np.int32, "SK_DPD_DEF": np.int32,
    "CNT_INSTALMENT": np.float32, "CNT_INSTALMENT_FUTURE": np.float32
}
pos = pd.read_csv(DATA_PATH + "POS_CASH_balance.csv", dtype=pos_dtype)
print("POS_CASH shape:", pos.shape)
# Load installments_payments.csv with compact dtypes
install_dtype = {
    "SK_ID_PREV": np.uint32, "SK_ID_CURR": np.uint32, "NUM_INSTALMENT_NUMBER": np.int32,
    "NUM_INSTALMENT_VERSION": np.float32, "DAYS_INSTALMENT": np.float32,
    "DAYS_ENTRY_PAYMENT": np.float32, "AMT_INSTALMENT": np.float32, "AMT_PAYMENT": np.float32
}
installments = pd.read_csv(DATA_PATH + "installments_payments.csv", dtype=install_dtype)
print("Installments shape:", installments.shape)
# Load credit_card_balance.csv with compact dtypes
card_dtype = {
    "SK_ID_PREV": np.uint32, "SK_ID_CURR": np.uint32, "MONTHS_BALANCE": np.int16,
    "AMT_CREDIT_LIMIT_ACTUAL": np.int32, "CNT_DRAWINGS_CURRENT": np.int32,
    "SK_DPD": np.int32, "SK_DPD_DEF": np.int32,
    "AMT_BALANCE": np.float32, "AMT_DRAWINGS_ATM_CURRENT": np.float32,
    "AMT_DRAWINGS_CURRENT": np.float32, "AMT_DRAWINGS_OTHER_CURRENT": np.float32,
    "AMT_DRAWINGS_POS_CURRENT": np.float32, "AMT_INST_MIN_REGULARITY": np.float32,
    "AMT_PAYMENT_CURRENT": np.float32, "AMT_PAYMENT_TOTAL_CURRENT": np.float32,
    "AMT_RECEIVABLE_PRINCIPAL": np.float32, "AMT_RECIVABLE": np.float32,
    "AMT_TOTAL_RECEIVABLE": np.float32, "CNT_DRAWINGS_ATM_CURRENT": np.float32,
    "CNT_DRAWINGS_OTHER_CURRENT": np.float32, "CNT_DRAWINGS_POS_CURRENT": np.float32,
    "CNT_INSTALMENT_MATURE_CUM": np.float32
}
credit_card = pd.read_csv(DATA_PATH + "credit_card_balance.csv", dtype=card_dtype)
print("Credit card shape:", credit_card.shape)


Application shape: (307511, 122)
Previous application shape: (1670214, 37)
Bureau shape: (1716428, 17)
Bureau balance shape: (27299925, 3)
POS_CASH shape: (10001358, 8)
Installments shape: (13605401, 8)
Credit card shape: (3840312, 23)


In [ ]:
# Reapply DAYS_EMPLOYED sentinel fix, EXT_SOURCE flags, ratios, EXT_SOURCE combos
application["DAYS_EMPLOYED_ANOM"] = (application["DAYS_EMPLOYED"] == 365243).astype(int)
application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, np.nan)

application["EXT_SOURCE_1_MISSING"] = application["EXT_SOURCE_1"].isna().astype(int)
application["EXT_SOURCE_3_MISSING"] = application["EXT_SOURCE_3"].isna().astype(int)

application["AGE_YEARS"] = -application["DAYS_BIRTH"] / 365.25
application["EMPLOYMENT_YEARS"] = -application["DAYS_EMPLOYED"] / 365.25

application["CREDIT_INCOME_RATIO"] = application["AMT_CREDIT"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_INCOME_RATIO"] = application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"]
application["ANNUITY_CREDIT_RATIO"] = application["AMT_ANNUITY"] / application["AMT_CREDIT"]
application["GOODS_CREDIT_RATIO"] = application["AMT_GOODS_PRICE"] / application["AMT_CREDIT"]
application["EMPLOYMENT_AGE_RATIO"] = application["EMPLOYMENT_YEARS"] / application["AGE_YEARS"]

ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
application["EXT_SOURCE_MEAN"] = application[ext_cols].mean(axis=1)
application["EXT_SOURCE_MIN"] = application[ext_cols].min(axis=1)
application["EXT_SOURCE_MAX"] = application[ext_cols].max(axis=1)
application["EXT_SOURCE_STD"] = application[ext_cols].std(axis=1)

application["EXT_SOURCE_1_2"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_2"]
application["EXT_SOURCE_1_3"] = application["EXT_SOURCE_1"] * application["EXT_SOURCE_3"]
application["EXT_SOURCE_2_3"] = application["EXT_SOURCE_2"] * application["EXT_SOURCE_3"]

print("MODEL02 application features recreated.")

MODEL02 application features recreated.


In [ ]:
# Rebuild previous_application aggregations: counts, financials, ratios, status rates, timing, payments
prev_count = (
    prev.groupby("SK_ID_CURR")
    .size()
    .rename("PREV_APPLICATION_COUNT")
    .reset_index()
)

financial_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT": ["mean", "max", "sum"],
        "AMT_APPLICATION": ["mean", "max", "sum"],
        "AMT_ANNUITY": ["mean", "max", "sum"],
        "AMT_GOODS_PRICE": ["mean", "max", "sum"],
        "AMT_DOWN_PAYMENT": ["mean", "max"],
        "RATE_DOWN_PAYMENT": ["mean", "max"]
    })
)
financial_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in financial_agg.columns]
financial_agg = financial_agg.reset_index()

prev["PREV_CREDIT_APPL_RATIO"] = prev["AMT_CREDIT"] / prev["AMT_APPLICATION"].replace(0, np.nan)
prev["PREV_CREDIT_APPL_DIFF"] = prev["AMT_CREDIT"] - prev["AMT_APPLICATION"]

relationship_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_CREDIT_APPL_RATIO": ["mean", "max"],
        "PREV_CREDIT_APPL_DIFF": ["mean", "max"]
    })
)
relationship_agg.columns = ["PREV_" + col[0] + "_" + col[1].upper() for col in relationship_agg.columns]
relationship_agg = relationship_agg.reset_index()

prev["PREV_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
prev["PREV_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev["PREV_CANCELED"] = (prev["NAME_CONTRACT_STATUS"] == "Canceled").astype(int)
prev["PREV_UNUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Unused offer").astype(int)

status_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({
        "PREV_APPROVED": "sum",
        "PREV_REFUSED": "sum",
        "PREV_CANCELED": "sum",
        "PREV_UNUSED": "sum"
    })
    .reset_index()
)
status_agg = status_agg.rename(columns={
    "PREV_APPROVED": "PREV_APPROVED_COUNT",
    "PREV_REFUSED": "PREV_REFUSED_COUNT",
    "PREV_CANCELED": "PREV_CANCELED_COUNT",
    "PREV_UNUSED": "PREV_UNUSED_COUNT"
})

status_agg = status_agg.merge(
    prev_count[["SK_ID_CURR", "PREV_APPLICATION_COUNT"]],
    on="SK_ID_CURR", how="left"
)
status_agg["PREV_APPROVAL_RATE"] = status_agg["PREV_APPROVED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_REFUSAL_RATE"] = status_agg["PREV_REFUSED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg["PREV_CANCELLATION_RATE"] = status_agg["PREV_CANCELED_COUNT"] / status_agg["PREV_APPLICATION_COUNT"]
status_agg = status_agg.drop(columns=["PREV_APPLICATION_COUNT"])

decision_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"DAYS_DECISION": ["min", "max", "mean"]})
)
decision_agg.columns = ["PREV_DAYS_DECISION_" + col[1].upper() for col in decision_agg.columns]
decision_agg = decision_agg.reset_index()

payment_agg = (
    prev.groupby("SK_ID_CURR")
    .agg({"CNT_PAYMENT": ["mean", "max", "sum"]})
)
payment_agg.columns = ["PREV_CNT_PAYMENT_" + col[1].upper() for col in payment_agg.columns]
payment_agg = payment_agg.reset_index()

prev_features = prev_count.copy()
for block in [financial_agg, relationship_agg, status_agg, decision_agg, payment_agg]:
    prev_features = prev_features.merge(block, on="SK_ID_CURR", how="left")

print("Previous-application features:", prev_features.shape)

Previous-application features: (338857, 35)


In [ ]:
# Rebuild bureau aggregations: counts, financials, overdue severity, status counts, credit types, timing
bureau_count = (
    bureau.groupby("SK_ID_CURR")
    .size()
    .rename("BUREAU_CREDIT_COUNT")
    .reset_index()
)

bureau_financial_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "AMT_CREDIT_SUM": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["mean", "max", "sum"],
        "AMT_CREDIT_SUM_LIMIT": ["mean", "max"],
        "AMT_ANNUITY": ["mean"]
    })
)
bureau_financial_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_financial_agg.columns]
bureau_financial_agg = bureau_financial_agg.reset_index()

bureau["BUREAU_OVERDUE_FLAG"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)

bureau_overdue_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "CREDIT_DAY_OVERDUE": ["max"],
        "BUREAU_OVERDUE_FLAG": ["sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["max", "sum"]
    })
    .reset_index()
)
bureau_overdue_agg.columns = [
    "SK_ID_CURR", "BUREAU_OVERDUE_DAYS_MAX", "BUREAU_OVERDUE_COUNT",
    "BUREAU_OVERDUE_AMOUNT_MAX", "BUREAU_OVERDUE_AMOUNT_SUM"
]

bureau_overdue_agg = bureau_overdue_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
bureau_overdue_agg["BUREAU_OVERDUE_RATIO"] = (
    bureau_overdue_agg["BUREAU_OVERDUE_COUNT"] / bureau_overdue_agg["BUREAU_CREDIT_COUNT"]
)
bureau_overdue_agg = bureau_overdue_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau["BUREAU_ACTIVE_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau["BUREAU_CLOSED_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Closed").astype(int)
bureau["BUREAU_SOLD_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Sold").astype(int)
bureau["BUREAU_BAD_DEBT_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Bad debt").astype(int)

bureau_status_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_ACTIVE_FLAG": "sum",
        "BUREAU_CLOSED_FLAG": "sum",
        "BUREAU_SOLD_FLAG": "sum",
        "BUREAU_BAD_DEBT_FLAG": "sum"
    })
    .reset_index()
)
bureau_status_agg = bureau_status_agg.rename(columns={
    "BUREAU_ACTIVE_FLAG": "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_FLAG": "BUREAU_CLOSED_COUNT",
    "BUREAU_SOLD_FLAG": "BUREAU_SOLD_COUNT",
    "BUREAU_BAD_DEBT_FLAG": "BUREAU_BAD_DEBT_COUNT"
})

bureau_status_agg = bureau_status_agg.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
bureau_status_agg["BUREAU_ACTIVE_RATIO"] = (
    bureau_status_agg["BUREAU_ACTIVE_COUNT"] / bureau_status_agg["BUREAU_CREDIT_COUNT"]
)
bureau_status_agg = bureau_status_agg.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau_type_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CREDIT_TYPE": "nunique"})
    .reset_index()
)
bureau_type_agg = bureau_type_agg.rename(columns={"CREDIT_TYPE": "BUREAU_CREDIT_TYPE_COUNT"})

bureau["BUREAU_CREDIT_CARD_FLAG"] = (bureau["CREDIT_TYPE"] == "Credit card").astype(int)
bureau["BUREAU_MORTGAGE_FLAG"] = (bureau["CREDIT_TYPE"] == "Mortgage").astype(int)
bureau["BUREAU_MICROLOAN_FLAG"] = (bureau["CREDIT_TYPE"] == "Microloan").astype(int)

bureau_type_specific = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "BUREAU_CREDIT_CARD_FLAG": "sum",
        "BUREAU_MORTGAGE_FLAG": "sum",
        "BUREAU_MICROLOAN_FLAG": "sum"
    })
    .reset_index()
)
bureau_type_specific = bureau_type_specific.rename(columns={
    "BUREAU_CREDIT_CARD_FLAG": "BUREAU_CREDIT_CARD_COUNT",
    "BUREAU_MORTGAGE_FLAG": "BUREAU_MORTGAGE_COUNT",
    "BUREAU_MICROLOAN_FLAG": "BUREAU_MICROLOAN_COUNT"
})

bureau_type_agg = bureau_type_agg.merge(bureau_type_specific, on="SK_ID_CURR", how="left")

bureau_timing_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({
        "DAYS_CREDIT": ["min", "max", "mean"],
        "DAYS_CREDIT_UPDATE": ["mean"]
    })
)
bureau_timing_agg.columns = ["BUREAU_" + col[0] + "_" + col[1].upper() for col in bureau_timing_agg.columns]
bureau_timing_agg = bureau_timing_agg.reset_index()

bureau_prolong_agg = (
    bureau.groupby("SK_ID_CURR")
    .agg({"CNT_CREDIT_PROLONG": "sum"})
    .reset_index()
)
bureau_prolong_agg = bureau_prolong_agg.rename(columns={"CNT_CREDIT_PROLONG": "BUREAU_CREDIT_PROLONG_TOTAL"})

bureau_features = bureau_count.copy()
for block in [bureau_financial_agg, bureau_overdue_agg, bureau_status_agg, bureau_type_agg, bureau_timing_agg, bureau_prolong_agg]:
    bureau_features = bureau_features.merge(block, on="SK_ID_CURR", how="left")

print("Bureau features:", bureau_features.shape)

Bureau features: (305811, 30)


In [ ]:
# Rebuild two-stage bureau_balance aggregation: monthly -> credit -> applicant
bureau_balance["BB_STATUS_0"] = (bureau_balance["STATUS"] == "0").astype(int)
bureau_balance["BB_STATUS_1"] = (bureau_balance["STATUS"] == "1").astype(int)
bureau_balance["BB_STATUS_2_PLUS"] = bureau_balance["STATUS"].isin(["2", "3", "4", "5"]).astype(int)
bureau_balance["BB_STATUS_C"] = (bureau_balance["STATUS"] == "C").astype(int)
bureau_balance["BB_STATUS_X"] = (bureau_balance["STATUS"] == "X").astype(int)

bb_credit = (
    bureau_balance.groupby("SK_ID_BUREAU")
    .agg({
        "MONTHS_BALANCE": ["count"],
        "BB_STATUS_0": "sum",
        "BB_STATUS_1": "sum",
        "BB_STATUS_2_PLUS": "sum",
        "BB_STATUS_C": "sum",
        "BB_STATUS_X": "sum"
    })
)
bb_credit.columns = ["BB_" + col[0] + "_" + col[1].upper() for col in bb_credit.columns]
bb_credit = bb_credit.reset_index()

bb_credit = bb_credit.rename(columns={
    "BB_BB_STATUS_0_SUM": "BB_STATUS_0_COUNT",
    "BB_BB_STATUS_1_SUM": "BB_STATUS_1_COUNT",
    "BB_BB_STATUS_2_PLUS_SUM": "BB_STATUS_2_PLUS_COUNT",
    "BB_BB_STATUS_C_SUM": "BB_STATUS_C_COUNT",
    "BB_BB_STATUS_X_SUM": "BB_STATUS_X_COUNT"
})

bb_credit["BB_DELINQUENCY_COUNT"] = bb_credit["BB_STATUS_1_COUNT"] + bb_credit["BB_STATUS_2_PLUS_COUNT"]
bb_credit["BB_DELINQUENCY_RATE"] = bb_credit["BB_DELINQUENCY_COUNT"] / bb_credit["BB_MONTHS_BALANCE_COUNT"]
bb_credit["BB_STATUS_2_PLUS_RATE"] = bb_credit["BB_STATUS_2_PLUS_COUNT"] / bb_credit["BB_MONTHS_BALANCE_COUNT"]
bb_credit["BB_EVER_DELINQUENT"] = (bb_credit["BB_DELINQUENCY_COUNT"] > 0).astype(int)
bb_credit["BB_EVER_SEVERE_DELINQUENCY"] = (bb_credit["BB_STATUS_2_PLUS_COUNT"] > 0).astype(int)

bureau_with_balance = bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].merge(bb_credit, on="SK_ID_BUREAU", how="left")

bb_app = (
    bureau_with_balance.groupby("SK_ID_CURR")
    .agg({
        "BB_MONTHS_BALANCE_COUNT": ["mean", "max", "sum"],
        "BB_DELINQUENCY_COUNT": ["mean", "max", "sum"],
        "BB_DELINQUENCY_RATE": ["mean", "max"],
        "BB_STATUS_2_PLUS_RATE": ["mean", "max"],
        "BB_STATUS_1_COUNT": "sum",
        "BB_STATUS_2_PLUS_COUNT": "sum",
        "BB_STATUS_C_COUNT": "sum",
        "BB_STATUS_X_COUNT": "sum",
        "BB_EVER_DELINQUENT": "sum",
        "BB_EVER_SEVERE_DELINQUENCY": "sum"
    })
)
bb_app.columns = ["BB_" + col[0] + "_" + col[1].upper() for col in bb_app.columns]
bb_app = bb_app.reset_index()
bb_app.columns = [col.replace("BB_BB_", "BB_") for col in bb_app.columns]

credits_with_balance = (
    bureau_with_balance.groupby("SK_ID_CURR")["BB_MONTHS_BALANCE_COUNT"]
    .count()
    .rename("BUREAU_CREDITS_WITH_BALANCE_HISTORY")
    .reset_index()
)
credits_with_balance = credits_with_balance.merge(
    bureau_count[["SK_ID_CURR", "BUREAU_CREDIT_COUNT"]],
    on="SK_ID_CURR", how="left"
)
credits_with_balance["BUREAU_BALANCE_COVERAGE_RATIO"] = (
    credits_with_balance["BUREAU_CREDITS_WITH_BALANCE_HISTORY"] / credits_with_balance["BUREAU_CREDIT_COUNT"]
)
credits_with_balance = credits_with_balance.drop(columns=["BUREAU_CREDIT_COUNT"])

bureau_balance_features = bb_app.merge(credits_with_balance, on="SK_ID_CURR", how="left")

print("Bureau balance features:", bureau_balance_features.shape)

Bureau balance features: (305811, 19)


In [ ]:
# Merge application + previous_application + bureau + bureau_balance
application_model = application.copy()
application_model = application_model.merge(prev_features, on="SK_ID_CURR", how="left")
application_model = application_model.merge(bureau_features, on="SK_ID_CURR", how="left")
application_model = application_model.merge(bureau_balance_features, on="SK_ID_CURR", how="left")

print("Model05 base shape:", application_model.shape)

Model05 base shape: (307511, 220)


In [ ]:
# Rebuild simplified POS_CASH features: overall aggregates, count, late rate, status share, completed rate
pos["LATE_POS"] = (pos["SK_DPD"] > 0).astype("int8")
pos["SK_DPD_RATIO"] = pos["SK_DPD"] / (pos["SK_DPD_DEF"] + 1)
pos["POS_REMAINING_INST_RATIO"] = pos["CNT_INSTALMENT_FUTURE"] / pos["CNT_INSTALMENT"].replace(0, np.nan)
pos["POS_INSTALLMENT_PROGRESS"] = 1 - pos["POS_REMAINING_INST_RATIO"]

pos_numeric_cols = [
    "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "SK_DPD_RATIO", "POS_REMAINING_INST_RATIO", "POS_INSTALLMENT_PROGRESS"
]
pos_overall = (
    pos.groupby("SK_ID_CURR")[pos_numeric_cols]
    .agg(["min", "max", "mean", "sum", "var"])
)
pos_overall.columns = ["POS_" + col[0] + "_" + col[1].upper() for col in pos_overall.columns]
pos_overall = pos_overall.reset_index()

pos_count = (
    pos.groupby("SK_ID_CURR")
    .size()
    .rename("POS_COUNT")
    .reset_index()
)

pos_late_rate = (
    pos.groupby("SK_ID_CURR")["LATE_POS"]
    .mean()
    .rename("POS_LATE_RATE")
    .reset_index()
)

status_dummies = pd.get_dummies(
    pos[["SK_ID_CURR", "NAME_CONTRACT_STATUS"]],
    columns=["NAME_CONTRACT_STATUS"], dummy_na=True, dtype=np.float32
)
status_share = status_dummies.groupby("SK_ID_CURR").mean().reset_index()
status_share_cols = [col for col in status_share.columns if col != "SK_ID_CURR"]
status_share = status_share.rename(columns={col: "POS_" + col for col in status_share_cols})

completed_rate = (
    pos.assign(COMPLETED=(pos["NAME_CONTRACT_STATUS"] == "Completed").astype(int))
    .groupby("SK_ID_CURR")["COMPLETED"]
    .mean()
    .rename("POS_COMPLETED_RATE")
    .reset_index()
)

pos_features = pos_overall.copy()
for block in [pos_count, pos_late_rate, status_share, completed_rate]:
    pos_features = pos_features.merge(block, on="SK_ID_CURR", how="left")

application_model = application_model.merge(pos_features, on="SK_ID_CURR", how="left")
application_model["HAS_POS_HISTORY"] = application_model["POS_COUNT"].notna().astype(int)

print("Model06 shape:", application_model.shape)

Model06 shape: (307511, 274)


In [ ]:
# Rebuild installment payment-behavior features
installments["INSTALLMENT_PAYMENT_RATIO"] = installments["AMT_PAYMENT"] / installments["AMT_INSTALMENT"].replace(0, np.nan)
installments["INSTALLMENT_PAYMENT_DIFF"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"]
installments["DAYS_PAYMENT_DELAY"] = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
installments["DAYS_PAYMENT_DELAY_POSITIVE"] = installments["DAYS_PAYMENT_DELAY"].clip(lower=0)
installments["DAYS_PAID_EARLY"] = (-installments["DAYS_PAYMENT_DELAY"]).clip(lower=0)
installments["LATE_PAYMENT"] = (installments["DAYS_PAYMENT_DELAY"] > 0).astype("int8")
installments["UNDERPAYMENT"] = (installments["INSTALLMENT_PAYMENT_DIFF"] > 0).astype("int8")

installment_agg = (
    installments.groupby("SK_ID_CURR")
    .agg({
        "NUM_INSTALMENT_VERSION": ["nunique"],
        "NUM_INSTALMENT_NUMBER": ["max", "mean"],
        "INSTALLMENT_PAYMENT_RATIO": ["mean", "max", "min"],
        "INSTALLMENT_PAYMENT_DIFF": ["mean", "max", "sum"],
        "DAYS_PAYMENT_DELAY_POSITIVE": ["mean", "max", "sum"],
        "DAYS_PAID_EARLY": ["mean", "max", "sum"],
        "AMT_INSTALMENT": ["mean", "max", "sum"],
        "AMT_PAYMENT": ["mean", "max", "sum"],
        "LATE_PAYMENT": ["sum"],
        "UNDERPAYMENT": ["sum"]
    })
)
installment_agg.columns = ["INST_" + col[0] + "_" + col[1].upper() for col in installment_agg.columns]
installment_agg = installment_agg.reset_index()

installment_count = (
    installments.groupby("SK_ID_CURR")
    .size()
    .rename("INST_INSTALLMENT_COUNT")
    .reset_index()
)
installment_late_rate = (
    installments.groupby("SK_ID_CURR")["LATE_PAYMENT"]
    .mean()
    .rename("INST_LATE_PAYMENT_RATE")
    .reset_index()
)
installment_underpayment_rate = (
    installments.groupby("SK_ID_CURR")["UNDERPAYMENT"]
    .mean()
    .rename("INST_UNDERPAYMENT_RATE")
    .reset_index()
)

for block in [installment_count, installment_late_rate, installment_underpayment_rate]:
    installment_agg = installment_agg.merge(block, on="SK_ID_CURR", how="left")

installment_feature_cols = [col for col in installment_agg.columns if col != "SK_ID_CURR"]

application_model = application_model.merge(installment_agg, on="SK_ID_CURR", how="left")
application_model["HAS_INSTALLMENT_HISTORY"] = application_model["INST_INSTALLMENT_COUNT"].notna().astype(int)

print("Model07 shape:", application_model.shape)

Model07 shape: (307511, 301)


In [ ]:
# Confirm shape, ID counts, and missingness before feature engineering
print("Credit card rows:", len(credit_card))
print("Unique applicants:", credit_card["SK_ID_CURR"].nunique())
print("Unique previous credit cards:", credit_card["SK_ID_PREV"].nunique())
print("Missing values:")
display(credit_card.isna().sum())

Credit card rows: 3840312
Unique applicants: 103558
Unique previous credit cards: 104307
Missing values:


,0
SK_ID_PREV,0
SK_ID_CURR,0
MONTHS_BALANCE,0
AMT_BALANCE,0
AMT_CREDIT_LIMIT_ACTUAL,0
AMT_DRAWINGS_ATM_CURRENT,749816
AMT_DRAWINGS_CURRENT,0
AMT_DRAWINGS_OTHER_CURRENT,749816
AMT_DRAWINGS_POS_CURRENT,749816
AMT_INST_MIN_REGULARITY,305236


In [ ]:
# Utilization, payment/receivable, drawing/limit, minimum-payment ratios, delinquency flags
credit_card["CC_UTILIZATION_RATIO"] = (
    credit_card["AMT_BALANCE"] / credit_card["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
)
credit_card["CC_PAYMENT_RECEIVABLE_RATIO"] = (
    credit_card["AMT_PAYMENT_TOTAL_CURRENT"] / credit_card["AMT_TOTAL_RECEIVABLE"].replace(0, np.nan)
)
credit_card["CC_DRAWING_LIMIT_RATIO"] = (
    credit_card["AMT_DRAWINGS_CURRENT"] / credit_card["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
)
credit_card["CC_MIN_PAYMENT_RATIO"] = (
    credit_card["AMT_INST_MIN_REGULARITY"] / credit_card["AMT_PAYMENT_CURRENT"].replace(0, np.nan)
)
credit_card["CC_LATE_FLAG"] = (credit_card["SK_DPD"] > 0).astype("int8")
credit_card["CC_SEVERE_DPD_FLAG"] = (credit_card["SK_DPD_DEF"] > 0).astype("int8")

In [ ]:
# Verify no infinite ratios and check zero-denominator prevalence for each ratio
ratio_cols = ["CC_UTILIZATION_RATIO", "CC_PAYMENT_RECEIVABLE_RATIO", "CC_DRAWING_LIMIT_RATIO", "CC_MIN_PAYMENT_RATIO"]

print("Infinite values in credit-card ratios:")
for col in ratio_cols:
    print(col, ":", np.isinf(credit_card[col]).sum())

print("\nZero credit limits:", (credit_card["AMT_CREDIT_LIMIT_ACTUAL"] == 0).sum())
print("Zero total receivable:", (credit_card["AMT_TOTAL_RECEIVABLE"] == 0).sum())
print("Zero current payment:", (credit_card["AMT_PAYMENT_CURRENT"] == 0).sum())

Infinite values in credit-card ratios:
CC_UTILIZATION_RATIO : 0
CC_PAYMENT_RECEIVABLE_RATIO : 0
CC_DRAWING_LIMIT_RATIO : 0
CC_MIN_PAYMENT_RATIO : 0

Zero credit limits: 753823
Zero total receivable: 2113643
Zero current payment: 390507


In [ ]:
# Applicant-level summary of balance, drawings, payments, receivables, and ratios
cc_numeric_features = [
    "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_CURRENT", "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_POS_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT",
    "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE",
    "AMT_INST_MIN_REGULARITY",
    "CNT_DRAWINGS_CURRENT", "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_POS_CURRENT", "CNT_DRAWINGS_OTHER_CURRENT",
    "CNT_INSTALMENT_MATURE_CUM", "SK_DPD", "SK_DPD_DEF",
    "CC_UTILIZATION_RATIO", "CC_PAYMENT_RECEIVABLE_RATIO", "CC_DRAWING_LIMIT_RATIO", "CC_MIN_PAYMENT_RATIO"
]

cc_overall = (
    credit_card.groupby("SK_ID_CURR")[cc_numeric_features]
    .agg(["min", "max", "mean", "sum"])
)
cc_overall.columns = ["CC_" + col[0] + "_" + col[1].upper() for col in cc_overall.columns]
cc_overall = cc_overall.reset_index()

display(cc_overall.head())

,SK_ID_CURR,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_MONTHS_BALANCE_MEAN,CC_MONTHS_BALANCE_SUM,CC_AMT_BALANCE_MIN,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_SUM,CC_AMT_CREDIT_LIMIT_ACTUAL_MIN,...,CC_CC_PAYMENT_RECEIVABLE_RATIO_MEAN,CC_CC_PAYMENT_RECEIVABLE_RATIO_SUM,CC_CC_DRAWING_LIMIT_RATIO_MIN,CC_CC_DRAWING_LIMIT_RATIO_MAX,CC_CC_DRAWING_LIMIT_RATIO_MEAN,CC_CC_DRAWING_LIMIT_RATIO_SUM,CC_CC_MIN_PAYMENT_RATIO_MIN,CC_CC_MIN_PAYMENT_RATIO_MAX,CC_CC_MIN_PAYMENT_RATIO_MEAN,CC_CC_MIN_PAYMENT_RATIO_SUM
0,100006,-6,-1,-3.5,-21,0.0,0.00000,0.000000,0.00,270000,...,NaN,0.000000,0.0,0.0,0.000000,0.000000,NaN,NaN,NaN,0.000000
1,100011,-75,-2,-38.5,-2849,0.0,189000.00000,54482.113281,4031676.25,90000,...,-2.59325,-95.950249,0.0,1.0,0.013514,1.000000,0.0,1.000,0.434935,31.750261
2,100013,-96,-1,-48.5,-4656,0.0,161420.21875,18159.919922,1743352.25,45000,...,0.15909,3.818152,0.0,1.0,0.037798,3.628571,0.0,0.875,0.076527,6.734413
3,100021,-18,-2,-10.0,-170,0.0,0.00000,0.000000,0.00,675000,...,NaN,0.000000,0.0,0.0,0.000000,0.000000,NaN,NaN,NaN,0.000000
4,100023,-11,-4,-7.5,-60,0.0,0.00000,0.000000,0.00,45000,...,NaN,0.000000,0.0,0.0,0.000000,0.000000,NaN,NaN,NaN,0.000000


In [ ]:
# Total record count, distinct card count, late/severe-DPD rates
cc_count = (
    credit_card.groupby("SK_ID_CURR")
    .size()
    .rename("CC_RECORD_COUNT")
    .reset_index()
)
cc_late_rate = (
    credit_card.groupby("SK_ID_CURR")["CC_LATE_FLAG"]
    .mean()
    .rename("CC_LATE_RATE")
    .reset_index()
)
cc_severe_late_rate = (
    credit_card.groupby("SK_ID_CURR")["CC_SEVERE_DPD_FLAG"]
    .mean()
    .rename("CC_SEVERE_DPD_RATE")
    .reset_index()
)
cc_card_count = (
    credit_card.groupby("SK_ID_CURR")["SK_ID_PREV"]
    .nunique()
    .rename("CC_CARD_COUNT")
    .reset_index()
)

In [ ]:
# Merge all credit card feature blocks together
cc_features = cc_overall.copy()
for block in [cc_count, cc_card_count, cc_late_rate, cc_severe_late_rate]:
    cc_features = cc_features.merge(block, on="SK_ID_CURR", how="left")

print("Credit-card feature table shape:", cc_features.shape)
print("Unique applicants:", cc_features["SK_ID_CURR"].nunique())
print("Duplicate applicants:", cc_features["SK_ID_CURR"].duplicated().sum())

Credit-card feature table shape: (103558, 101)
Unique applicants: 103558
Duplicate applicants: 0


In [ ]:
# List every engineered credit-card feature
cc_feature_cols = [col for col in cc_features.columns if col != "SK_ID_CURR"]

print("Number of credit-card features:", len(cc_feature_cols))
for col in cc_feature_cols:
    print("-", col)

Number of credit-card features: 100
- CC_MONTHS_BALANCE_MIN
- CC_MONTHS_BALANCE_MAX
- CC_MONTHS_BALANCE_MEAN
- CC_MONTHS_BALANCE_SUM
- CC_AMT_BALANCE_MIN
- CC_AMT_BALANCE_MAX
- CC_AMT_BALANCE_MEAN
- CC_AMT_BALANCE_SUM
- CC_AMT_CREDIT_LIMIT_ACTUAL_MIN
- CC_AMT_CREDIT_LIMIT_ACTUAL_MAX
- CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN
- CC_AMT_CREDIT_LIMIT_ACTUAL_SUM
- CC_AMT_DRAWINGS_CURRENT_MIN
- CC_AMT_DRAWINGS_CURRENT_MAX
- CC_AMT_DRAWINGS_CURRENT_MEAN
- CC_AMT_DRAWINGS_CURRENT_SUM
- CC_AMT_DRAWINGS_ATM_CURRENT_MIN
- CC_AMT_DRAWINGS_ATM_CURRENT_MAX
- CC_AMT_DRAWINGS_ATM_CURRENT_MEAN
- CC_AMT_DRAWINGS_ATM_CURRENT_SUM
- CC_AMT_DRAWINGS_POS_CURRENT_MIN
- CC_AMT_DRAWINGS_POS_CURRENT_MAX
- CC_AMT_DRAWINGS_POS_CURRENT_MEAN
- CC_AMT_DRAWINGS_POS_CURRENT_SUM
- CC_AMT_DRAWINGS_OTHER_CURRENT_MIN
- CC_AMT_DRAWINGS_OTHER_CURRENT_MAX
- CC_AMT_DRAWINGS_OTHER_CURRENT_MEAN
- CC_AMT_DRAWINGS_OTHER_CURRENT_SUM
- CC_AMT_PAYMENT_CURRENT_MIN
- CC_AMT_PAYMENT_CURRENT_MAX
- CC_AMT_PAYMENT_CURRENT_MEAN
- CC_AMT_PAYMENT_CURR

In [ ]:
# Add credit card features onto the running Model07 dataset
application_model = application_model.merge(cc_features, on="SK_ID_CURR", how="left")

print("Shape after credit-card features:", application_model.shape)
print("Unique applicants:", application_model["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", application_model["SK_ID_CURR"].duplicated().sum())

Shape after credit-card features: (307511, 401)
Unique applicants: 307511
Duplicate applicant IDs: 0


In [ ]:
# Create the has-history flag post-merge, as the single source of truth
application_model["HAS_CREDIT_CARD_HISTORY"] = application_model["CC_CARD_COUNT"].notna().astype(int)

print(application_model["HAS_CREDIT_CARD_HISTORY"].value_counts())

HAS_CREDIT_CARD_HISTORY
0    220606
1     86905
Name: count, dtype: int64


In [ ]:
# Split into X and y for modeling
X = application_model.drop(columns=["TARGET", "SK_ID_CURR"])
y = application_model["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 400)
y shape: (307511,)


In [ ]:
# Same stratified split parameters used throughout the project
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training shape: (246008, 400)
Validation shape: (61503, 400)

Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [ ]:
# Separate numeric vs categorical columns for preprocessing
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 384
Categorical features: 16


In [ ]:
# Median-impute numerics, most-frequent-impute + one-hot encode categoricals
numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
# Same hyperparameters as every prior experiment for a fair feature-only comparison
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [ ]:
import gc

# Delete raw datasets no longer needed
del installments
del credit_card
del pos
del bureau_balance
del prev
del bureau

gc.collect()

print("Garbage collection complete.")

Garbage collection complete.


In [ ]:
# Combine preprocessing and model
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

In [ ]:
# Fit the pipeline on training data
print("Training XGBoost with Model07 + Credit Card features...")
xgb_pipeline.fit(X_train, y_train)
print("Training complete.")

Training XGBoost with Model07 + Credit Card features...
Training complete.


In [ ]:
# Generate validation-set probability predictions
valid_proba = xgb_pipeline.predict_proba(X_valid)[:, 1]
print("Predictions generated.")

Predictions generated.


In [ ]:
# Compute ROC-AUC and PR-AUC on the validation set
roc_auc = roc_auc_score(y_valid, valid_proba)
pr_auc = average_precision_score(y_valid, valid_proba)

print("XGBOOST + MODEL07 + CREDIT CARD")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

XGBOOST + MODEL07 + CREDIT CARD
ROC-AUC: 0.7870
PR-AUC:  0.2872


In [ ]:
# Measure improvement over the locked Model07 result
MODEL07_ROC_AUC = 0.7859
MODEL07_PR_AUC = 0.2864

roc_change = roc_auc - MODEL07_ROC_AUC
pr_change = pr_auc - MODEL07_PR_AUC

print("IMPROVEMENT OVER MODEL07")
print(f"ROC-AUC change: {roc_change:+.4f}")
print(f"PR-AUC change:  {pr_change:+.4f}")

IMPROVEMENT OVER MODEL07
ROC-AUC change: +0.0011
PR-AUC change:  +0.0008


In [ ]:
# Build a summary table of this experiment's results
credit_card_result = pd.DataFrame({
    "Experiment": ["XGBoost + Model07 + Credit Card Features"],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "ROC-AUC Change": [roc_change],
    "PR-AUC Change": [pr_change]
})

display(credit_card_result.style.format({
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}",
    "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change
0,XGBoost + Model07 + Credit Card Features,0.7870,0.2872,+0.0011,+0.0008


In [ ]:
# Save this experiment's result to CSV
credit_card_result.to_csv(RESULTS_PATH + "credit_card_experiment.csv", index=False)
print("Experiment saved to:", RESULTS_PATH + "credit_card_experiment.csv")

Experiment saved to: /content/drive/MyDrive/RupeeRisk/credit_card_experiment.csv


In [ ]:
# Install MLflow in this Colab session
!pip install mlflow -q
import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [ ]:
# Point at the same tracking database used throughout the project
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")
# Record this run's parameters and metrics
with mlflow.start_run(run_name="XGBoost_Credit_Card"):
    mlflow.log_param("stage", "Model08 - Credit Card Feature Engineering")
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("builds_on", "Model07 application + previous + bureau + bureau balance + POS + installments")
    mlflow.log_param("n_credit_card_features", len(cc_feature_cols))
    mlflow.log_param("derived_ratios", "utilization, payment_receivable, drawing_limit, minimum_payment")
    mlflow.log_param("scale_pos_weight", False)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("roc_auc_change_vs_Model07", roc_change)
    mlflow.log_metric("pr_auc_change_vs_Model07", pr_change)

print("Model08 logged to MLflow.")

Model08 logged to MLflow.


In [ ]:
# Pull every run logged so far for comparison
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(runs[["tags.mlflow.runName", "metrics.roc_auc", "metrics.pr_auc"]])

,tags.mlflow.runName,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Credit_Card,0.787019,0.287171
1,XGBoost_Installments,0.785949,0.286362
2,XGBoost_POS_CASH,0.783313,0.278581
3,XGBoost_Bureau,0.777585,0.274161
4,XGBoost_Previous_Application,0.775428,0.265853
5,XGBoost_Application_Features,0.769403,0.262725
6,XGBoost_scale_pos_weight,0.760000,0.249300
7,XGBoost_Baseline,0.761200,0.251600
8,Logistic_Regression_Baseline,0.750100,0.232600


In [ ]:
# List every credit-card feature created in this notebook
print("Credit-card features created:")
for col in cc_feature_cols:
    print("-", col)
print("\nTotal credit-card features:", len(cc_feature_cols))

Credit-card features created:
- CC_MONTHS_BALANCE_MIN
- CC_MONTHS_BALANCE_MAX
- CC_MONTHS_BALANCE_MEAN
- CC_MONTHS_BALANCE_SUM
- CC_AMT_BALANCE_MIN
- CC_AMT_BALANCE_MAX
- CC_AMT_BALANCE_MEAN
- CC_AMT_BALANCE_SUM
- CC_AMT_CREDIT_LIMIT_ACTUAL_MIN
- CC_AMT_CREDIT_LIMIT_ACTUAL_MAX
- CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN
- CC_AMT_CREDIT_LIMIT_ACTUAL_SUM
- CC_AMT_DRAWINGS_CURRENT_MIN
- CC_AMT_DRAWINGS_CURRENT_MAX
- CC_AMT_DRAWINGS_CURRENT_MEAN
- CC_AMT_DRAWINGS_CURRENT_SUM
- CC_AMT_DRAWINGS_ATM_CURRENT_MIN
- CC_AMT_DRAWINGS_ATM_CURRENT_MAX
- CC_AMT_DRAWINGS_ATM_CURRENT_MEAN
- CC_AMT_DRAWINGS_ATM_CURRENT_SUM
- CC_AMT_DRAWINGS_POS_CURRENT_MIN
- CC_AMT_DRAWINGS_POS_CURRENT_MAX
- CC_AMT_DRAWINGS_POS_CURRENT_MEAN
- CC_AMT_DRAWINGS_POS_CURRENT_SUM
- CC_AMT_DRAWINGS_OTHER_CURRENT_MIN
- CC_AMT_DRAWINGS_OTHER_CURRENT_MAX
- CC_AMT_DRAWINGS_OTHER_CURRENT_MEAN
- CC_AMT_DRAWINGS_OTHER_CURRENT_SUM
- CC_AMT_PAYMENT_CURRENT_MIN
- CC_AMT_PAYMENT_CURRENT_MAX
- CC_AMT_PAYMENT_CURRENT_MEAN
- CC_AMT_PAYMENT_CURRENT_SU

In [ ]:
# Print the overall before/after comparison for this stage
print("""
Model08 CREDIT CARD FEATURE ENGINEERING COMPLETE

Model07 benchmark: ROC-AUC = 0.7859, PR-AUC = 0.2864
Model08 result:    ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Change:            ROC-AUC = {:+.4f}, PR-AUC = {:+.4f}

Next: consolidate feature blocks, feature redundancy/selection,
polynomial experiment, Optuna, SHAP, final evaluation.
""".format(roc_auc, pr_auc, roc_change, pr_change))


Model08 CREDIT CARD FEATURE ENGINEERING COMPLETE

Model07 benchmark: ROC-AUC = 0.7859, PR-AUC = 0.2864
Model08 result:    ROC-AUC = 0.7870, PR-AUC = 0.2872
Change:            ROC-AUC = +0.0011, PR-AUC = +0.0008

Next: consolidate feature blocks, feature redundancy/selection,
polynomial experiment, Optuna, SHAP, final evaluation.



In [ ]:
# ============================================================
#  SAVE MODEL08 CONSOLIDATED DATASET
# ============================================================

MODEL08_DATA_PATH = (
    "/content/drive/MyDrive/RupeeRisk/"
    "model08_consolidated_features.parquet"
)

application_model.to_parquet(
    MODEL08_DATA_PATH,
    index=False
)

print(
    "Model08 consolidated dataset saved to:",
    MODEL08_DATA_PATH
)

print(
    "Shape:",
    application_model.shape
)

Model08 consolidated dataset saved to: /content/drive/MyDrive/RupeeRisk/model08_consolidated_features.parquet
Shape: (307511, 402)


In [ ]:
# Re-read the file to make sure it was written correctly.

model08_check = pd.read_parquet(
    MODEL08_DATA_PATH
)

print(
    "Loaded-back shape:",
    model08_check.shape
)

print(
    "Applicants:",
    model08_check["SK_ID_CURR"].nunique()
)

print(
    "Duplicate applicant IDs:",
    model08_check["SK_ID_CURR"].duplicated().sum()
)

print(
    "TARGET present:",
    "TARGET" in model08_check.columns
)

print(
    "Parquet verification successful."
)

del model08_check

gc.collect()

Loaded-back shape: (307511, 402)
Applicants: 307511
Duplicate applicant IDs: 0
TARGET present: True
Parquet verification successful.


93

In [ ]:
# CELL 47 — SAVE MODEL08 FEATURE LIST
# ============================================================

MODEL08_FEATURE_LIST_PATH = (
    "/content/drive/MyDrive/RupeeRisk/"
    "model08_feature_list.csv"
)

model08_features = [
    col
    for col in application_model.columns
    if col not in [
        "TARGET",
        "SK_ID_CURR"
    ]
]

pd.DataFrame({
    "Feature": model08_features
}).to_csv(
    MODEL08_FEATURE_LIST_PATH,
    index=False
)

print(
    "Model08 feature list saved to:",
    MODEL08_FEATURE_LIST_PATH
)

print(
    "Total Model08 features:",
    len(model08_features)
)

Model08 feature list saved to: /content/drive/MyDrive/RupeeRisk/model08_feature_list.csv
Total Model08 features: 400


In [ ]:
# CELL 48 — SAVE MODEL08 BENCHMARK
# ============================================================

MODEL08_BENCHMARK_PATH = (
    "/content/drive/MyDrive/RupeeRisk/"
    "model08_benchmark.csv"
)

model08_benchmark = pd.DataFrame({

    "Model": [
        "Model08"
    ],

    "ROC-AUC": [
        0.7870
    ],

    "PR-AUC": [
        0.2872
    ]
})

model08_benchmark.to_csv(
    MODEL08_BENCHMARK_PATH,
    index=False
)

print(
    "Model08 benchmark saved to:",
    MODEL08_BENCHMARK_PATH
)

Model08 benchmark saved to: /content/drive/MyDrive/RupeeRisk/model08_benchmark.csv
